# 🗺️ Day 21 · OpenStreetMap: old city, new city

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2021%20OpenStreetMap/tutorial/day21_old_city_new_city.ipynb)

**OpenStreetMap** (OSM) is a map of the whole world made by volunteers, like a Wikipedia for maps. Anyone can add a road, a shop or a house, and anyone can download the data for free.

Today you download every **building** around three places and compare them:
- **Khiva**, the old walled city of Ichan-Kala, with small courtyard houses packed together,
- **Chilanzar** in Tashkent, with long Soviet apartment blocks,
- **Tashkent City**, with new towers and malls.

In this notebook you will:
1. download buildings from OSM with **osmnx**,
2. measure every building in **m²** (and see why degrees are no good for that),
3. colour the buildings by size,
4. compare the three places on the same scale, like fingerprints,
5. find out which buildings OSM still knows little about, and how you can fix that,
6. map your own neighbourhood and save a poster.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

**osmnx** downloads data from OpenStreetMap. It asks a free server called **Overpass**: "give me everything tagged `building` within 700 m of this point".

Overpass is shared by the whole world, so sometimes it is busy. Our helper `get_buildings` waits at most 90 seconds. If the server doesn't answer, it uses a **saved copy** of the same data that we downloaded in November 2026.

In [ ]:
!pip -q install osmnx

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from concurrent.futures import ThreadPoolExecutor

BASE = "https://raw.githubusercontent.com/NodiraTillayeva/30DayMapChallengeUzbekistan/main/2026/Day%2021%20OpenStreetMap/tutorial/data/"
RADIUS = 700        # metres around each point

PLACES = {
    "Khiva":         (41.3785, 60.3600),
    "Chilanzar":     (41.2760, 69.2040),
    "Tashkent City": (41.3155, 69.2490),
}
SAVED = {
    "Khiva":         "khiva.geojson",
    "Chilanzar":     "chilanzar.geojson",
    "Tashkent City": "tashkent_city.geojson",
}

ox.settings.requests_timeout = 60
ox.settings.overpass_rate_limit = False

def get_buildings(lat, lng, saved=None, wait=90):
    job = ThreadPoolExecutor(1).submit(
        ox.features_from_point, (lat, lng), tags={"building": True}, dist=RADIUS)
    try:
        b = job.result(timeout=wait)
        print("downloaded live from OpenStreetMap ✅")
    except Exception:
        if saved is None:
            raise RuntimeError("The OSM server is busy. Wait a few minutes and try again.")
        b = gpd.read_file(BASE + saved)
        print("the OSM server is busy, so we used the saved copy 💾")
    b = b[b.geom_type.isin(["Polygon", "MultiPolygon"])]
    keep = [c for c in ["building", "name", "building:levels"] if c in b.columns]
    return b[keep + ["geometry"]].reset_index(drop=True)

print("Ready! ✅ osmnx version", ox.__version__)

## Step 1 · Download the buildings of Khiva

In OSM, everything is described by **tags**: a key and a value, like `building=house` or `building=mosque`. We ask for everything that has the key `building`, whatever its value.

In [ ]:
lat, lng = PLACES["Khiva"]
buildings = get_buildings(lat, lng, SAVED["Khiva"])

print(len(buildings), "buildings")
buildings.head(10)

Each row is one building. `building` says what kind it is, and `name` is filled in only for famous ones. `geometry` is its outline: a **polygon**, a list of corners in longitude and latitude.

Let's draw them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
buildings.plot(ax=ax, color="#333333")
ax.set_aspect(1.3)
ax.axis("off")
plt.show()

## Step 2 · Degrees are not metres

Our outlines are in **degrees** of longitude and latitude (`EPSG:4326`). If you ask for the area now, you get tiny numbers in "square degrees", which mean nothing: a degree of longitude is about 84 km in Khiva but about 0 km at the North Pole.

So first we **project** the buildings to metres. `estimate_utm_crs` picks the right **UTM** zone for this place (Khiva is in zone 41, Tashkent in zone 42). Then `.area` gives m².

In [ ]:
print("in degrees:", buildings.geometry.iloc[0].area)

buildings = buildings.to_crs(buildings.estimate_utm_crs())
buildings["area"] = buildings.area

print("in metres: ", round(buildings["area"].iloc[0]), "m²")
print("projection:", buildings.crs.name)

Now we can sort. Here are the biggest buildings around Ichan-Kala that have a name. Do you know them? Look at `Ichan Qal'a devori`: someone mapped the city wall as a building!

In [ ]:
named = buildings.dropna(subset=["name"])
named.sort_values("area", ascending=False)[["building", "name", "area"]].head(8).round()

## Step 3 · Colour the buildings by size

We put every building into one of five size groups with `pd.cut`. A small house in the old city is about 100 m², a Soviet block about 1,000 m², a mall more than 10,000 m².

To make all three places look alike, we cut a **circle** of 700 m around the point with `.clip`.

In [ ]:
BINS   = [0, 100, 300, 1000, 3000, np.inf]
LABELS = ["under 100 m²", "100–300 m²", "300–1,000 m²", "1,000–3,000 m²", "over 3,000 m²"]
COLORS = ["#7fb7e6", "#5fa8a8", "#e9c46a", "#f08a4b", "#e63946"]
BACK   = "#0f1720"

def prepare(b, lat, lng):
    b = b.to_crs(b.estimate_utm_crs())
    centre = gpd.GeoSeries(gpd.points_from_xy([lng], [lat]), crs=4326).to_crs(b.crs)
    circle = centre.buffer(RADIUS).iloc[0]
    b = b.clip(circle)
    b = b[b.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    b["area"] = b.area
    b["size"] = pd.cut(b["area"], BINS, labels=LABELS)
    return b, circle

def draw(ax, b, circle):
    ax.set_facecolor(BACK)
    gpd.GeoSeries([circle], crs=b.crs).plot(ax=ax, color="#17222d")
    for label, color in zip(LABELS, COLORS):
        b[b["size"] == label].plot(ax=ax, color=color, linewidth=0)
    ax.set_xlim(circle.bounds[0] - 20, circle.bounds[2] + 20)
    ax.set_ylim(circle.bounds[1] - 20, circle.bounds[3] + 20)
    ax.set_aspect("equal")
    ax.axis("off")

khiva, circle = prepare(buildings, lat, lng)

fig, ax = plt.subplots(figsize=(10, 8), facecolor=BACK)
draw(ax, khiva, circle)
legend = [Patch(color=c, label=l) for l, c in zip(LABELS, COLORS)]
ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(1, 1), frameon=False, labelcolor="white", fontsize=10)
plt.show()

print(khiva["size"].value_counts().reindex(LABELS))

## Step 4 · Three places, one scale

Now the same for all three. Every circle is **1.4 km across**, so you can compare them directly.

In [ ]:
city = {}
for name, (lat, lng) in PLACES.items():
    print(name, end=": ")
    b = get_buildings(lat, lng, SAVED[name])
    city[name] = prepare(b, lat, lng)

fig, axes = plt.subplots(1, 3, figsize=(18, 6.6), facecolor=BACK)
for ax, (name, (b, circle)) in zip(axes, city.items()):
    draw(ax, b, circle)
    ax.set_title(name, color="white", fontsize=14)
fig.legend(handles=legend, loc="lower center", bbox_to_anchor=(0.5, 0.03), ncol=5, frameon=False, labelcolor="white", fontsize=11)
plt.show()

## Step 5 · Three fingerprints

Maps are good for the eye, numbers are good for comparing. For each place we count the buildings, find the **median** size (half the buildings are smaller, half are bigger) and the share of the circle covered by roofs.

In [ ]:
rows = []
for name, (b, circle) in city.items():
    rows.append({
        "place": name,
        "buildings": len(b),
        "median m²": round(b["area"].median()),
        "biggest m²": round(b["area"].max()),
        "% covered": round(b["area"].sum() / circle.area * 100, 1),
    })
table = pd.DataFrame(rows).set_index("place")
table

A **histogram** shows how many buildings there are of each size. Sizes go from 10 m² to 50,000 m², so we use a **log scale**: each step on the x axis is 10 times bigger than the one before.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
bins = np.logspace(1, 5, 41)
for ax, (name, (b, circle)) in zip(axes, city.items()):
    n, edges, bars = ax.hist(b["area"], bins=bins)
    for bar, left in zip(bars, edges):
        bar.set_color(COLORS[np.searchsorted(BINS, left, side="right") - 1])
    ax.set_xscale("log")
    ax.set_ylabel(name)
    ax.spines[["top", "right"]].set_visible(False)
axes[-1].set_xlabel("building size, m²")
plt.show()

## Step 6 · What kind of building is it?

The tag `building=yes` means "there is a building here, but nobody said what kind". Mappers often draw the outline first, from a satellite picture, and the type comes later from someone who has been there.

Let's count the tags in each place.

In [ ]:
for name, (b, circle) in city.items():
    print(name)
    print(b["building"].value_counts().head(6).to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6.6), facecolor=BACK)
for ax, (name, (b, circle)) in zip(axes, city.items()):
    ax.set_facecolor(BACK)
    gpd.GeoSeries([circle], crs=b.crs).plot(ax=ax, color="#17222d")
    unknown = b["building"] == "yes"
    b[unknown].plot(ax=ax, color="#5b6875", linewidth=0)
    b[~unknown].plot(ax=ax, color="#e9c46a", linewidth=0)
    share = unknown.mean() * 100
    ax.set_title(f"{name}: {share:.0f}% just 'yes'", color="white", fontsize=13)
    ax.set_aspect("equal")
    ax.axis("off")
plt.show()

Grey buildings are only `building=yes`. Yellow ones have a type: `house`, `apartments`, `school`, `mosque`…

**You can fix this.** OSM is made by people like you:
1. Go to [openstreetmap.org](https://www.openstreetmap.org) and create a free account.
2. Zoom to a place you know well, like your street or your school, and click **Edit**.
3. Click a building and choose its type: *House*, *Apartment building*, *School*…, then **Save**.

Only add what you know from being there or from the satellite picture inside the editor. Don't copy from Google Maps or Yandex Maps: their data isn't free to use.

## Step 7 · Your own neighbourhood

Put in the latitude and longitude of your house or school. To find them, right-click the place on [openstreetmap.org](https://www.openstreetmap.org) and choose **Show address**.

This time there is no saved copy, so if the server is busy, wait a few minutes and run the cell again.

In [ ]:
my_lat, my_lng = 40.3842, 71.7843     # Fergana, the centre

mine = get_buildings(my_lat, my_lng)
mine, my_circle = prepare(mine, my_lat, my_lng)

fig, ax = plt.subplots(figsize=(8, 8), facecolor=BACK)
draw(ax, mine, my_circle)
plt.show()
print(len(mine), "buildings, median", round(mine["area"].median()), "m²")

## Step 8 · Save your poster

The three circles side by side, with only their names, saved as **old_city_new_city.png**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6.8), facecolor=BACK)
for ax, (name, (b, circle)) in zip(axes, city.items()):
    draw(ax, b, circle)
    ax.text(0.5, -0.02, name.upper(), transform=ax.transAxes, ha="center", va="top",
            color="#8a97a3", fontsize=11, family="monospace")
plt.subplots_adjust(wspace=0.04, left=0.01, right=0.99, top=0.99, bottom=0.06)
plt.savefig("old_city_new_city.png", dpi=200, facecolor=BACK)
plt.show()

try:
    from google.colab import files
    files.download("old_city_new_city.png")
except ImportError:
    print("Saved old_city_new_city.png")

## 🎉 Done!

You downloaded OpenStreetMap buildings, measured them in m², and compared three ways of building a city.

Share your map with **#30DayMapChallenge** and **#30DayMapChallengeUzbekistan**, and please credit:
> Map data © OpenStreetMap contributors (ODbL).